In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Train over Original Dataset

In [2]:
df_class = pd.read_csv(r"df_main_classi_noise_original_headers.csv")

In [3]:
# split train and test data
from sklearn.model_selection import train_test_split

train, test = train_test_split(df_class, test_size=0.2, random_state=42)

In [4]:
from sklearn.preprocessing import StandardScaler, scale
import statsmodels.api as sm

# train data
X_train = train.drop(columns=['healthcare_access'])
# X_names = X
y_train = train['healthcare_access']

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = sm.add_constant(X_train)

# test data
X_test = test.drop(columns=['healthcare_access'])
# X_names = X
y_test = test['healthcare_access']

X_test = scaler.transform(X_test)
X_test = sm.add_constant(X_test)


### Logistic Regression Models

In [7]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
np.random.seed(42)

strengths = [1e0 ,1e1, 1e2, 1e3, 1e4]
rates = [0.2,0.4,0.5, 0.6,0.8]
#l1 validation 

results1 = {}
results2 = {}
print('elasticnet')
for strength in strengths:
    for rate in rates:
        print(strength)
        model = LogisticRegression(penalty = 'elasticnet', l1_ratio = rate, C = strength, solver = 'saga', max_iter = 10000)
        model.fit(X_train,y_train) 
        avg_score = model.score(X_train,y_train)
        results1[strength] = avg_score
        results2[rate] = avg_score


elasticnet
1.0
1.0
1.0
1.0
1.0
10.0
10.0
10.0
10.0
10.0
100.0
100.0
100.0
100.0
100.0
1000.0
1000.0
1000.0
1000.0
1000.0
10000.0
10000.0
10000.0


KeyboardInterrupt: 

In [ ]:
# results2 = {}
# print('l2')
# for strength in strengths:
#     print(strength)
#     model = LogisticRegression(penalty = 'l2', C = strength, solver = 'lbfgs', max_iter = 100000)
#     model.fit(X_train,y_train) 
#     avg_score = model.score(X_test,y_test)
#     results2[strength] = avg_score

In [8]:
best_strength_l1 = max(results1, key = results1.get) # gamma = 100
best_strength_l2 = max(results2, key = results2.get) # gamma = 1000

print(best_strength_l1, best_strength_l2) # 100000  1

100.0 0.2


# Train over 100K Dataset

In [9]:
df_100k = pd.read_csv("df_main_classi_noise_100k_headers.csv")


In [10]:
# split train and test data
from sklearn.model_selection import train_test_split

train, test = train_test_split(df_100k, test_size=0.2, random_state=42)

In [11]:

# train data
X_train = train.drop(columns=['avg_death_rate', 'healthcare_access'])
# X_names = X
y_train = train['healthcare_access']

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = sm.add_constant(X_train)

# test data
X_test = test.drop(columns=['avg_death_rate','healthcare_access'])
# X_names = X
y_test = test['healthcare_access']

X_test = scaler.transform(X_test)
X_test = sm.add_constant(X_test)


In [13]:
# best Logistic Regression L2 model 

l2_model = LogisticRegression(penalty = 'elasticnet', l1_ratio = 0.2, C = 100, solver = 'saga', max_iter = 10000)
l2_model.fit(X_train,y_train)
avg_score = l2_model.score(X_test,y_test)

# ran 4 mins

In [14]:
print(avg_score)

0.7136640992297689


In [15]:
# important features

# # calculates the weights of each feature
# weights_l2 = l2_model.coef_[0]*(X_train.max(axis=0)-X_train.min(axis = 0))/X_train.std(axis=0)
# weights_l1 = l1_model.coef_[0]*(X_train.max(axis=0)-X_train.min(axis = 0))/X_train.std(axis=0)

from sklearn.inspection import permutation_importance

# compute importances
model_fi = permutation_importance(l2_model, X_train, y_train)
model_fi['importances_mean']

array([ 0.00000000e+00,  1.08410233e-02,  1.11536249e-02,  5.77687749e-03,
        2.04566484e-03,  2.97346638e-03,  4.26388576e-03,  1.17263111e-02,
        7.75251957e-05,  4.00130042e-04,  1.39545352e-03,  6.75219446e-05,
        1.05284217e-03,  5.00162553e-06,  7.75251957e-04,  4.52647110e-04,
        6.37707255e-04,  3.45112161e-04,  1.80558682e-03,  1.88061120e-03,
       -6.75219446e-05,  1.87560957e-03,  1.72556081e-04,  4.50146298e-05,
        1.45047140e-04,  1.75056893e-05,  5.27671493e-04,  2.02565834e-04,
        7.75251957e-05,  1.12536574e-04,  6.75219446e-05,  2.50081276e-06,
        4.80156051e-04, -2.82591842e-04,  5.70185310e-04,  1.10035762e-04,
       -1.07534949e-04,  8.57778778e-04,  2.31075099e-03, -1.12536574e-04,
        1.80308600e-03,  9.00292595e-05,  5.50178808e-05,  4.75154425e-05,
        7.22484808e-03,  3.65118664e-03,  5.21869608e-02,  8.03136019e-02,
        6.75219446e-05,  3.37359642e-03,  3.66369070e-03,  2.81841599e-03,
        3.54865331e-03,  

In [18]:
sorted_idxl2 = model_fi['importances_mean'].argsort()[::-1]

X_df = train.drop(columns=['avg_death_rate','healthcare_access'])

flag = 0
for i in sorted_idxl2:
    if flag < 20:
        flag = flag+1
        print(f"{X_df.columns[i]}:{model_fi['importances_mean'][i]}")

pct_pop_commutes_by_taxi_motorcycle_or_other:0.0803136019206242
pct_pop_commutes_by_public_transport_subway:0.052186960762247735
households_cohabiting_couple:0.0411058594043064
households_with_annual_income_75000_to_99999:0.03773726461099858
households_with_annual_income_50000_to_59999:0.03298572035911671
households_married_couple:0.03271313176782453
pct_households_cohabiting_couple:0.03233300822767397
pop_has_3_or_more_vehicles_available:0.026871233150774
households_with_computing_devices_desktop_laptop:0.025768374721784594
pop_commute_departure_0800_to_0829:0.02492560082026658
pop_works_occupation_management_business_science_arts:0.02383524645509789
households_with_annual_income_150000_to_199999:0.023332583089504078
pop_sex_male_age_65_and_above:0.023145022132192983
pop_total:0.021949633630930054
pop_adult_education_less_than_high_school:0.02175707104809066
pop_commute_travel_time_20_to_24_min:0.02066671668292195
pop_works_industry_educational_healthcare_social_assistance:0.019123715